#修了課題①　Titanic

利用するデータセット：タイタニック号の搭乗者データ

合格基準：テストデータに対して正解率80%以上

データセットの中身

*   pclass： 旅客クラス（1＝1等、2＝2等、3＝3等）。(裕福さの目安)
*   name： 乗客の名前
*   sex： 性別（male＝男性、female＝女性）
*   age： 年齢。一部の乳児は小数値
*   sibsp： タイタニック号に同乗している兄弟（Siblings）や配偶者（Spouses）の数
*   parch： タイタニック号に同乗している親（Parents）や子供（Children）の数
*   ticket： チケット番号
*   fare： 旅客運賃
*   cabin： 客室番号
*   embarked： 出港地（C＝Cherbourg：シェルブール、Q＝Queenstown：クイーンズタウン、S＝Southampton：サウサンプトン）
*   boat： 救命ボート番号
*   body： 遺体収容時の識別番号
*   home.dest： 自宅または目的地



# データのダウンロード
※変更しない様、お願いいたします。

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:

# 学習用のデータのダウンロード
!wget 'https://drive.google.com/uc?export=download&id=1-12Pg5IsjNAEbgk7G4a2WqFaOhpYmbyJ' -O titanic_train.csv
# 検証用のデータのダウンロード
!wget 'https://drive.google.com/uc?export=download&id=1jmzmYNPRWUGLcHeqhcwKTGlb_d2Rzn4Y' -O titanic_validation.csv

In [ ]:
# 変更しないようお願いいたします。
# 課題提出時に利用するデータのダウンロード
!wget 'https://drive.google.com/uc?export=download&id=1-1KX2NQmwUOXAstOE7c2INC1rRMNABVZ' -O titanic_test_x.csv

In [ ]:
# 学習用データ
train_df = pd.read_csv('titanic_train.csv', index_col=0)
display(train_df.head(3))

In [ ]:
# 検証用データ
val_df = pd.read_csv('titanic_validation.csv', index_col=0)
display(val_df.head(3))

In [ ]:
# 学習用データと検証用データの情報
# 欠損値がないようにみえるが、？で埋められているため処理が必要。
# objectデータは数値に変換する必要がある。
train_df = train_df.replace('?', np.nan)
val_df = val_df.replace('?', np.nan)
train_df.info()
val_df.info()

In [ ]:
# 課題提出時に利用するテストデータの読み込みと表示
X_test = pd.read_csv('titanic_test_x.csv', index_col=0)
X_test.head(3)

In [ ]:
X_test = X_test.replace('?', np.nan)
X_test.info()

#モデルの制作

In [ ]:
# ここからダウンロードされたデータに対してモデルを作成し、検証用データに対して予測結果を出力して下さい。

In [ ]:
# LightGBMのパラメータ
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

categorical_features = ['sex', 'embarked']
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    out = df[features].copy()
    out['sex'] = out['sex'].map({'male': 0, 'female': 1})
    out['embarked'] = out['embarked'].fillna('S')
    out['embarked'] = out['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    out['age'] = pd.to_numeric(out['age'], errors='coerce')
    out['age'] = out['age'].fillna(out['age'].median())
    out['fare'] = pd.to_numeric(out['fare'], errors='coerce')
    out['fare'] = out['fare'].fillna(out['fare'].median())
    return out

X_train = preprocess(train_df)
y_train = train_df['survived'].astype(int)

X_valid = preprocess(val_df)
y_valid = val_df['survived'].astype(int)

X_test = preprocess(X_test)

lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical_features)
lgb_valid = lgb.Dataset(X_valid, y_valid, reference=lgb_train, categorical_feature=categorical_features)

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=200,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(False)]
)

valid_pred = model.predict(X_valid, num_iteration=model.best_iteration)
valid_pred_label = (valid_pred >= 0.5).astype(int)
accuracy = np.mean(valid_pred_label == y_valid)
print(f'Validation accuracy: {accuracy:.4f}')

test_pred = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = pd.DataFrame((test_pred >= 0.5).astype(int), columns=['survived'], index=X_test.index)

# 課題提出

In [ ]:
# 提出形式見本
# 提出データの形、タイトル
# 309行、カラム名が「survived」になるようにする。
# 以前のセルで作成した y_pred をそのまま使います。

# csv形式で提出する。
y_pred.to_csv('y_pred.csv', index=True)
y_pred